# 04 - CasADi as a Formulation Tool for MPC

We already know the MPC formulation.
Now we express the same problem more cleanly with CasADi.
        

In [ ]:
%matplotlib inline

import time
import numpy as np
import matplotlib.pyplot as plt
import scipy.sparse as sp
import osqp
import casadi as ca
from scipy.linalg import solve_discrete_are, block_diag
from scipy.optimize import minimize

np.set_printoptions(precision=4, suppress=True)

if not ca.has_nlpsol('ipopt'):
    raise RuntimeError('CasADi IPOPT backend is not available in this environment. Install CasADi with IPOPT support.')

print('CasADi version:', ca.__version__)
        

## 1) Same scalar finite-horizon problem as in 02

Start with one known expression in NumPy, then write the same expression symbolically in CasADi.
        

In [ ]:
# Scalar benchmark
x0_known = 2.0
u0_known = -0.7

# Numeric expression (NumPy)
J_numpy = x0_known**2 + u0_known**2 + (x0_known + u0_known)**2
print('NumPy expression value:', J_numpy)

# Same expression (CasADi)
x0_sym = ca.SX.sym('x0')
u0_sym = ca.SX.sym('u0')
J_sym_expr = x0_sym**2 + u0_sym**2 + (x0_sym + u0_sym)**2
J_fun = ca.Function('J_fun', [x0_sym, u0_sym], [J_sym_expr])
print('CasADi expression value:', float(J_fun(x0_known, u0_known)))
        

## 2) Scalar MPC in CasADi Opti

Decision variables: U and X.
Parameter: x0.
Constraints: dynamics and input bounds.
Cost: stage terms plus terminal cost.
        

In [ ]:
# Scalar MPC setup
N_s = 8
a = 1.0
b = 1.0
q = 1.0
r = 1.0
u_max_s = 1.0

p_inf_s = 0.5 * (q + np.sqrt(q**2 + 4.0 * q * r))

opti_s = ca.Opti()
U_s = opti_s.variable(N_s)
X_s = opti_s.variable(N_s + 1)
x0_par_s = opti_s.parameter()

opti_s.subject_to(X_s[0] == x0_par_s)
for k in range(N_s):
    opti_s.subject_to(X_s[k + 1] == a * X_s[k] + b * U_s[k])
    opti_s.subject_to(opti_s.bounded(-u_max_s, U_s[k], u_max_s))

J_s = 0
for k in range(N_s):
    J_s += q * X_s[k] ** 2 + r * U_s[k] ** 2
J_s += p_inf_s * X_s[N_s] ** 2

opti_s.minimize(J_s)
opti_s.solver('ipopt', {'print_time': False}, {'print_level': 0, 'sb': 'yes'})

x0_test_s = 3.5
opti_s.set_value(x0_par_s, x0_test_s)
sol_s = opti_s.solve()

u0_casadi_s = float(sol_s.value(U_s[0]))
print(f'Scalar CasADi MPC first input: {u0_casadi_s:+.6f}')
        

## 3) Compare scalar CasADi result with manual numeric solve
        

In [ ]:
def scalar_manual_objective(U, x0):
    x = x0
    J = 0.0
    for uk in U:
        J += q * x**2 + r * uk**2
        x = a * x + b * uk
    J += p_inf_s * x**2
    return J

U0 = np.zeros(N_s)
bounds = [(-u_max_s, u_max_s)] * N_s
res = minimize(lambda U: scalar_manual_objective(U, x0_test_s), U0, method='SLSQP', bounds=bounds, options={'maxiter': 300, 'ftol': 1e-10, 'disp': False})
if not res.success:
    raise RuntimeError(f'Manual scalar solve failed: {res.message}')

u0_manual_s = float(res.x[0])
print(f'Scalar manual first input:        {u0_manual_s:+.6f}')
print(f'Difference:                       {abs(u0_manual_s - u0_casadi_s):.3e}')
assert np.isclose(u0_manual_s, u0_casadi_s, atol=2e-5)
print('Check passed: scalar CasADi and manual solutions agree.')
        

## 4) Double-integrator MPC in CasADi

Use the same benchmark and notation as notebook 02.
        

In [ ]:
A = np.array([[1.0, 1.0],
              [0.0, 1.0]])
B = np.array([[0.0],
              [1.0]])
Q = np.diag([4.0, 1.0])
R = np.array([[0.2]])
P_inf = solve_discrete_are(A, B, Q, R)
K_inf = np.linalg.solve(R + B.T @ P_inf @ B, B.T @ P_inf @ A)

N = 10
nx = 2
nu = 1
u_max = 0.8
x_max = 4.8
v_max = 2.5

print('P_inf =')
print(P_inf)
print('K_inf =')
print(K_inf)
        

## 5) Manual condensed QP helper from 02 logic
        

In [ ]:
def build_prediction(A, B, N):
    nx = A.shape[0]
    nu = B.shape[1]
    Phi = np.zeros((nx * N, nx))
    Gamma = np.zeros((nx * N, nu * N))
    for i in range(1, N + 1):
        Phi[(i - 1) * nx : i * nx, :] = np.linalg.matrix_power(A, i)
        for j in range(i):
            Gamma[(i - 1) * nx : i * nx, j * nu : (j + 1) * nu] = np.linalg.matrix_power(A, i - 1 - j) @ B
    return Phi, Gamma


def solve_manual_qp_first_input(x0):
    Phi, Gamma = build_prediction(A, B, N)
    Qbar = block_diag(*([Q for _ in range(N - 1)] + [P_inf]))
    Rbar = block_diag(*([R for _ in range(N)]))

    H = Gamma.T @ Qbar @ Gamma + Rbar
    h = Gamma.T @ Qbar @ (Phi @ x0)
    P_solver = 2.0 * H
    q_solver = 2.0 * h

    G_u = np.vstack([np.eye(nu * N), -np.eye(nu * N)])
    g_u = u_max * np.ones(2 * nu * N)

    F_single = np.array([[1.0, 0.0], [-1.0, 0.0], [0.0, 1.0], [0.0, -1.0]])
    f_single = np.array([x_max, x_max, v_max, v_max])
    F_x = block_diag(*([F_single] * N))
    f_x = np.tile(f_single, N)

    A_state = F_x @ Gamma
    b_state = f_x - F_x @ (Phi @ x0)

    A_ineq = np.vstack([G_u, A_state])
    b_ineq = np.concatenate([g_u, b_state])

    prob = osqp.OSQP()
    prob.setup(
        P=sp.csc_matrix(0.5 * (P_solver + P_solver.T)),
        q=q_solver,
        A=sp.csc_matrix(A_ineq),
        l=-np.inf * np.ones(A_ineq.shape[0]),
        u=b_ineq,
        verbose=False,
        polish=True,
        eps_abs=1e-8,
        eps_rel=1e-8,
    )
    res = prob.solve()
    if res.info.status != 'solved':
        raise RuntimeError(f'Manual QP failed: {res.info.status}')
    return float(res.x[0])
        

## 6) CasADi Opti formulation for double integrator

Optional terminal constraint (after geometry notebook):
x_N^T P_inf x_N <= alpha_star.
        

In [ ]:
# Compute alpha_star from face constraints (state box + terminal controller input limit).
Krow = K_inf.reshape(-1)
F_faces = np.array([
    [1.0, 0.0],
    [-1.0, 0.0],
    [0.0, 1.0],
    [0.0, -1.0],
    Krow,
    -Krow,
])
b_faces = np.array([x_max, x_max, v_max, v_max, u_max, u_max])
Pinv = np.linalg.inv(P_inf)
alpha_star = float(0.995 * np.min([(bi**2) / (Fi @ Pinv @ Fi.T) for Fi, bi in zip(F_faces, b_faces)]))

use_terminal_constraint = True
print('alpha_star =', alpha_star)

opti = ca.Opti()
X = opti.variable(nx, N + 1)
U = opti.variable(nu, N)
x0_par = opti.parameter(nx)

opti.subject_to(X[:, 0] == x0_par)

for k in range(N):
    xk = X[:, k]
    uk = U[:, k]

    opti.subject_to(X[:, k + 1] == ca.mtimes(A, xk) + ca.mtimes(B, uk))
    opti.subject_to(opti.bounded(-u_max, uk[0], u_max))
    opti.subject_to(opti.bounded(-x_max, xk[0], x_max))
    opti.subject_to(opti.bounded(-v_max, xk[1], v_max))

xN = X[:, N]
opti.subject_to(opti.bounded(-x_max, xN[0], x_max))
opti.subject_to(opti.bounded(-v_max, xN[1], v_max))
if use_terminal_constraint:
    opti.subject_to(ca.mtimes([xN.T, P_inf, xN]) <= alpha_star)

J = 0
for k in range(N):
    xk = X[:, k]
    uk = U[:, k]
    J += ca.mtimes([xk.T, Q, xk]) + ca.mtimes([uk.T, R, uk])
J += ca.mtimes([xN.T, P_inf, xN])

opti.minimize(J)
opti.solver('ipopt', {'print_time': False}, {'print_level': 0, 'sb': 'yes'})
        

## 7) Compare first input: manual condensed QP vs CasADi Opti
        

In [ ]:
x0_compare = np.array([3.0, 1.1])

u0_manual = solve_manual_qp_first_input(x0_compare)
opti.set_value(x0_par, x0_compare)
sol = opti.solve()
u0_casadi = float(sol.value(U[0, 0]))

print(f'Manual QP first input   = {u0_manual:+.8f}')
print(f'CasADi Opti first input = {u0_casadi:+.8f}')
print(f'Absolute difference     = {abs(u0_manual - u0_casadi):.3e}')
assert np.isclose(u0_manual, u0_casadi, atol=2e-4)
print('Check passed: CasADi and manual QP first inputs agree.')
        

## 8) Receding-horizon simulation with CasADi formulation

At each step we solve, apply U[:,0], simulate one step, and resolve.
        

In [ ]:
steps = 22
x_sim = np.zeros((steps + 1, nx))
nu_sim = np.zeros(steps)
solve_ms = []

x_sim[0] = np.array([3.0, 1.1])

for k in range(steps):
    opti.set_value(x0_par, x_sim[k])
    t0 = time.perf_counter()
    sol = opti.solve()
    solve_ms.append(1000.0 * (time.perf_counter() - t0))

    u0 = float(sol.value(U[0, 0]))
    nu_sim[k] = u0

    x_next = A @ x_sim[k] + B[:, 0] * u0
    x_sim[k + 1] = x_next

fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
ax[0].plot(x_sim[:, 0], 'o-', label='position')
ax[0].axhline(x_max, color='r', linestyle=':')
ax[0].axhline(-x_max, color='r', linestyle=':')
ax[0].set_title('CasADi MPC position')
ax[0].set_xlabel('k')
ax[0].grid(True, alpha=0.3)
ax[0].legend()

ax[1].plot(x_sim[:, 1], 'o-', label='velocity')
ax[1].axhline(v_max, color='r', linestyle=':')
ax[1].axhline(-v_max, color='r', linestyle=':')
ax[1].set_title('CasADi MPC velocity')
ax[1].set_xlabel('k')
ax[1].grid(True, alpha=0.3)
ax[1].legend()

ax[2].step(range(steps), nu_sim, where='post', label='u')
ax[2].axhline(u_max, color='r', linestyle=':', label='u bounds')
ax[2].axhline(-u_max, color='r', linestyle=':')
ax[2].set_title('CasADi MPC input')
ax[2].set_xlabel('k')
ax[2].grid(True, alpha=0.3)
ax[2].legend()

plt.tight_layout()
plt.show()

print('Average CasADi solve time (ms):', np.mean(solve_ms))
        

## Takeaway

CasADi changed how we wrote the optimization problem, not the control concept.
The dynamics, cost, constraints, and receding-horizon logic stayed the same as in manual MPC.
        